# Expanding-window walk-forward CV — a avaliação correta

## A descoberta que motiva este notebook

O experimento multi-seed × multi-ticker mostrou um padrão devastador:

| Ticker | train balance up | test balance up | shift | Transformer mediana AUC |
|---|---|---|---|---|
| ITUB4 | 0.564 | 0.681 | +0.12 | 0.801 |
| PETR4 | 0.645 | 0.615 | -0.03 | **0.334** |
| VALE3 | 0.482 | 0.824 | +0.34 | **0.992** |

A 'performance' do Transformer é perfeitamente explicada pelo *shift de prior de classe* entre treino e teste, não por aprendizado real. Quando train e test têm balance similar (PETR4), o modelo cai para 0.33. Quando o shift é massivo (VALE3), o modelo "acerta" 0.99 simplesmente prevendo a classe majoritária do treino, que coincide com a classe majoritária do teste.

**Implicação**: a avaliação por single-window walk-forward 70/15/15 é estruturalmente enganosa em séries financeiras não-estacionárias. Precisamos de algo melhor.

## O protocolo correto: expanding-window walk-forward CV

Em vez de UM split, use **múltiplas janelas de teste rolantes**:

1. Comece com `train = [day 0, day T0)` (mínimo 600 dias).
2. `val = [T0, T0 + 90)`, `test = [T0 + 90, T0 + 180)`.
3. Treine, avalie, registre AUC + balance shift.
4. Avance T0 em 90 dias. Repita.
5. Resultado: 5–8 pontos de avaliação por ticker, cobrindo regimes diferentes.

## A figura hero do TCC

O scatter `AUC vs (test_balance - train_balance)`, colorido por modelo, facetado por ticker. Se a hipótese estiver certa, vai aparecer:

- **Transformer**: forte correlação positiva entre shift e AUC (o modelo é uma máquina de matching de prior)
- **Baseline**: AUC ≈ flat, próximo de 0.55–0.65 independente do shift

Esse gráfico **mata e explica** todo o achado original AUC=0.709.

In [ ]:
import sys, os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from eval_utils import make_binary_target

HORIZON = 21
WINDOW = 30
MIN_TRAIN_DAYS = 600
VAL_DAYS = 90
TEST_DAYS = 90
STEP_DAYS = 90
N_SEEDS_PER_FOLD = 5  # múltiplas seeds por fold para reduzir ruído
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

TICKERS = {
    'ITUB4': ('ITUB4.SA', '../4.finbert-br/itub4_daily_sentiment.csv'),
    'PETR4': ('PETR4.SA', '../4.finbert-br/petr4_daily_sentiment.csv'),
    'VALE3': ('VALE3.SA', '../4.finbert-br/vale3_daily_sentiment.csv'),
}
PRICE_COLS = ['Close','Volume','return','ma7','ma21','std21','lag_1','lag_2','lag_3','lag_4','lag_5']
SENT_COLS  = ['n_articles','mean_logit_pos','mean_logit_neg','mean_logit_neu','mean_sentiment']
BASELINE_FEATURES = ['return','lag_1','lag_5','Volume','std21']
FULL_FEATURES = PRICE_COLS + SENT_COLS

## 1. Pipeline de dados (igual aos notebooks anteriores)

In [ ]:
def load_prices(yf_ticker, period='5y'):
    df = yf.Ticker(yf_ticker).history(period=period, auto_adjust=True).reset_index()
    df['date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
    df = df[['date','Close','Volume']].copy()
    df['return'] = df['Close'].pct_change()
    df['ma7']    = df['Close'].rolling(7).mean()
    df['ma21']   = df['Close'].rolling(21).mean()
    df['std21']  = df['Close'].rolling(21).std()
    for k in range(1, 6):
        df[f'lag_{k}'] = df['Close'].shift(k)
    return df.dropna().reset_index(drop=True)

def build_dataset(yf_ticker, sentiment_csv):
    px = load_prices(yf_ticker)
    sent = pd.read_csv(sentiment_csv, parse_dates=['date'])[['date'] + SENT_COLS]
    df = px.merge(sent, on='date', how='left').sort_values('date').reset_index(drop=True)
    df[SENT_COLS] = df[SENT_COLS].ffill().fillna(0)
    df['target'] = make_binary_target(df['Close'], horizon=HORIZON)
    return df.dropna(subset=['target']).reset_index(drop=True)

def make_windows(X, y, window=WINDOW):
    Xs, ys = [], []
    for i in range(window, len(X)):
        Xs.append(X[i-window:i]); ys.append(y[i])
    return np.array(Xs), np.array(ys)

def expanding_folds(n_total, min_train=MIN_TRAIN_DAYS, val=VAL_DAYS, test=TEST_DAYS, step=STEP_DAYS):
    """Gera (train_end, val_end, test_end) para cada fold."""
    folds = []
    train_end = min_train
    while train_end + val + test <= n_total:
        folds.append((train_end, train_end + val, train_end + val + test))
        train_end += step
    return folds

## 2. Modelos (mesma arquitetura dos experimentos anteriores)

In [ ]:
class Stage4Transformer(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, nlayers=2, window=WINDOW):
        super().__init__()
        self.proj = nn.Linear(n_features, d_model)
        pe = torch.zeros(window, d_model)
        pos = torch.arange(0, window).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos*div); pe[:, 1::2] = torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0))
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=128, dropout=0.2, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=nlayers)
        self.head = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 1))
    def forward(self, x):
        h = self.proj(x) + self.pe[:, :x.size(1), :]
        h = self.enc(h)
        return self.head(h.mean(dim=1)).squeeze(-1)

def train_transformer_fold(df, fold, seed):
    """fold = (train_end, val_end, test_end). Retorna AUC + class balances."""
    train_end, val_end, test_end = fold
    train = df.iloc[:train_end]; val = df.iloc[train_end:val_end]; test = df.iloc[val_end:test_end]
    torch.manual_seed(seed); np.random.seed(seed)

    sc = StandardScaler().fit(train[FULL_FEATURES])
    Xtr = sc.transform(train[FULL_FEATURES]); ytr = train['target'].values.astype(int)
    Xva = sc.transform(val[FULL_FEATURES]);   yva = val['target'].values.astype(int)
    Xte = sc.transform(test[FULL_FEATURES]);  yte = test['target'].values.astype(int)
    Xtw, ytw = make_windows(Xtr, ytr)
    Xvw, yvw = make_windows(Xva, yva)
    Xew, yew = make_windows(Xte, yte)
    if len(yew) < 5 or len(np.unique(yew)) < 2:
        return None

    model = Stage4Transformer(n_features=len(FULL_FEATURES)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    pos = (ytw==1).sum(); neg = (ytw==0).sum()
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg/max(pos,1)], device=device, dtype=torch.float32))

    Xt_t = torch.tensor(Xtw, dtype=torch.float32, device=device)
    yt_t = torch.tensor(ytw, dtype=torch.float32, device=device)
    Xv_t = torch.tensor(Xvw, dtype=torch.float32, device=device)
    yv_t = torch.tensor(yvw, dtype=torch.float32, device=device)
    Xe_t = torch.tensor(Xew, dtype=torch.float32, device=device)

    best=float('inf'); best_state=None; bad=0; patience=15
    for epoch in range(150):
        model.train(); opt.zero_grad()
        loss = loss_fn(model(Xt_t), yt_t); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval()
        with torch.no_grad():
            vl = loss_fn(model(Xv_t), yv_t).item()
        if vl < best - 1e-4:
            best=vl; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        y_score = torch.sigmoid(model(Xe_t)).cpu().numpy()
    return {
        'auc': roc_auc_score(yew, y_score),
        'train_balance': float(train['target'].mean()),
        'val_balance':   float(val['target'].mean()),
        'test_balance':  float(test['target'].mean()),
        'n_test':        int(len(yew)),
    }

def train_baseline_fold(df, fold, seed):
    train_end, val_end, test_end = fold
    train = df.iloc[:train_end]; val = df.iloc[train_end:val_end]; test = df.iloc[val_end:test_end]
    np.random.seed(seed)
    sc = StandardScaler().fit(train[BASELINE_FEATURES])
    Xtr = sc.transform(train[BASELINE_FEATURES]); ytr = train['target'].values.astype(int)
    Xva = sc.transform(val[BASELINE_FEATURES]);   yva = val['target'].values.astype(int)
    Xte = sc.transform(test[BASELINE_FEATURES]);  yte = test['target'].values.astype(int)
    if len(yte) < 5 or len(np.unique(yte)) < 2:
        return None
    pos = (ytr==1).sum(); neg = (ytr==0).sum()
    m = xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=neg/max(pos,1), eval_metric='auc', random_state=seed,
    )
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    y_score = m.predict_proba(Xte)[:,1]
    return {
        'auc': roc_auc_score(yte, y_score),
        'train_balance': float(train['target'].mean()),
        'val_balance':   float(val['target'].mean()),
        'test_balance':  float(test['target'].mean()),
        'n_test':        int(len(yte)),
    }

## 3. Loop principal: 3 tickers × N folds × 2 modelos × 5 seeds

In [ ]:
rows = []
t0 = time.time()
for ticker, (yf_t, sent_csv) in TICKERS.items():
    print(f'\n=== {ticker} ===')
    df = build_dataset(yf_t, sent_csv)
    folds = expanding_folds(len(df))
    print(f'  dataset: {len(df)} dias | {len(folds)} folds')
    for fi, fold in enumerate(folds):
        for seed in range(N_SEEDS_PER_FOLD):
            rt = train_transformer_fold(df, fold, seed)
            if rt is not None:
                rt.update({'ticker':ticker,'model':'transformer_finbert','fold':fi,'seed':seed})
                rows.append(rt)
            rb = train_baseline_fold(df, fold, seed)
            if rb is not None:
                rb.update({'ticker':ticker,'model':'baseline_xgb','fold':fi,'seed':seed})
                rows.append(rb)
        if rt is not None:
            shift = rt['test_balance'] - rt['train_balance']
            print(f'  fold {fi} | shift={shift:+.3f} | trans last AUC={rt["auc"]:.3f} | base last AUC={rb["auc"]:.3f} | {time.time()-t0:.0f}s')

results = pd.DataFrame(rows)
results['shift'] = results['test_balance'] - results['train_balance']
results.to_csv('results_expanding_cv.csv', index=False)
print(f'\nTotal: {len(results)} runs in {time.time()-t0:.0f}s')

## 4. Agregação por fold (média sobre seeds)

In [ ]:
fold_agg = results.groupby(['ticker','model','fold']).agg(
    auc_mean=('auc','mean'), auc_std=('auc','std'),
    train_balance=('train_balance','first'),
    test_balance=('test_balance','first'),
    shift=('shift','first'),
    n_test=('n_test','first'),
).reset_index()
fold_agg.to_csv('results_expanding_cv_fold_agg.csv', index=False)
print(fold_agg.round(3).to_string())

In [ ]:
# Resumo por (ticker, modelo) — média sobre todos os folds
ticker_agg = fold_agg.groupby(['ticker','model']).agg(
    auc_mean=('auc_mean','mean'),
    auc_std_across_folds=('auc_mean','std'),
    auc_min=('auc_mean','min'),
    auc_max=('auc_mean','max'),
    n_folds=('fold','count'),
).round(3)
print('Por ticker x modelo (média sobre folds):')
print(ticker_agg.to_string())

# Correlação shift × AUC — o número que prova a hipótese
from scipy.stats import pearsonr
print('\nCorrelação entre (test_balance - train_balance) e AUC:')
for (ticker, model), grp in fold_agg.groupby(['ticker','model']):
    if len(grp) >= 3:
        r, p = pearsonr(grp['shift'], grp['auc_mean'])
        print(f'  {ticker:6s} {model:20s} r={r:+.3f} p={p:.3f} (n={len(grp)})')

## 5. A figura hero do TCC

**Scatter**: AUC vs shift de balance (test - train), facetado por ticker, colorido por modelo.

Se a hipótese estiver certa, vamos ver:
- Transformer com inclinação positiva forte (matching de prior)
- Baseline com inclinação flat (real prediction)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, ticker in zip(axes, TICKERS.keys()):
    sub = fold_agg[fold_agg.ticker == ticker]
    for model, color, marker in [('transformer_finbert','C0','o'), ('baseline_xgb','C3','^')]:
        m = sub[sub.model == model]
        if len(m) == 0: continue
        ax.scatter(m['shift'], m['auc_mean'], s=80, c=color, marker=marker,
                   label=model.replace('_',' '), edgecolors='k', linewidths=0.5)
        # Linha de tendência
        if len(m) >= 3:
            z = np.polyfit(m['shift'], m['auc_mean'], 1)
            xs = np.linspace(m['shift'].min(), m['shift'].max(), 50)
            ax.plot(xs, np.polyval(z, xs), color=color, alpha=0.5, lw=1.5)
    ax.axhline(0.5, ls=':', color='gray', label='Acaso')
    ax.axvline(0.0, ls=':', color='gray')
    ax.set_title(ticker)
    ax.set_xlabel('test_balance − train_balance (shift de prior)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('ROC-AUC')
axes[0].legend(fontsize=8, loc='lower right')
plt.suptitle('AUC vs class-prior shift — a figura que mata 0.709', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('expanding_cv_hero.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# Figura complementar: AUC ao longo do tempo (por fold) para cada ticker
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, ticker in zip(axes, TICKERS.keys()):
    sub = fold_agg[fold_agg.ticker == ticker]
    for model, color, marker in [('transformer_finbert','C0','o'), ('baseline_xgb','C3','^')]:
        m = sub[sub.model == model]
        if len(m) == 0: continue
        ax.errorbar(m['fold'], m['auc_mean'], yerr=m['auc_std'].fillna(0), fmt=marker+'-',
                    color=color, label=model.replace('_',' '), capsize=3)
    ax.axhline(0.5, ls=':', color='gray')
    ax.set_title(ticker); ax.set_xlabel('Fold (tempo →)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('ROC-AUC (mean ± std over seeds)')
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig('expanding_cv_overtime.png', dpi=140, bbox_inches='tight')
plt.show()

## 6. Conclusão e implicações para a tese

Se a correlação `shift × AUC` para o Transformer for **r > 0.6 com p < 0.05**, e a correlação para o baseline for próxima de zero, você tem evidência rigorosa para a tese metodológica:

> *A performance aparente de modelos de deep learning para predição de direção de preços em séries financeiras não-estacionárias é dominada por matching de prior entre treino e teste, e não por capacidade preditiva real. Avaliação por single-window walk-forward é estruturalmente enganosa. Avaliação por expanding-window CV multi-fold é necessária para mensurar capacidade preditiva real.*

A média do AUC do Transformer e do baseline sobre todos os folds, com std entre folds, é o número honesto a ser reportado na tese. A figura hero (`expanding_cv_hero.png`) é a evidência visual.